# 8. The full Arequipa DEM

Every number this project has published comes from **crops** — Colca, and small Arequipa
windows. The full DEM is the run that has never been done, and this notebook is where it
lands.

[Notebook 7](07_explaining_a_run.ipynb) covers how to drive the pipeline and how to read
what it says. This one is about one specific run, at a scale the crops cannot speak
for.

In [1]:
import json
import os

from oroscope import explain

**Read, not run.** The cells below open results that were produced locally and stored in
`results/arequipa_full/`. They do not start a search. Each of these searches takes about
half an hour, CI executes notebooks on every push, and a tutorial costing ninety minutes
of compute per commit is a bill rather than a tutorial. The expensive half runs once, on
a machine that has the DEM; the notebook opens a few hundred kilobytes of JSON.

To produce or refresh the store:

```bash
python tools/run_arequipa_full.py --dry-run   # report the cost, then stop
python tools/run_arequipa_full.py             # GRAND, TAMBO, then the combination
```

**Start with `--dry-run`.** It begins nothing — no search, no file, no change to the
store — and prints the five things worth knowing before committing an hour of a
machine:

```text
DEM:       input/dem/arequipa_SRTMGL1.tif
estimate:  2.32 GiB at downsample_factor 4
available: 5.4 GiB
would run: grand, tambo, then combine
expected:  ~25-30 minutes each
store:     results/arequipa_full
```

`DEM` says whether the file is even present, so a missing DEM is reported before the
first search starts rather than after. `estimate` against `available` is what decides
`downsample_factor` — the same DEM needs 4.5 GiB at 1 and 2.3 GiB at 4, since the
labelling arrays scale as its inverse square. `would run` honours `--only`, so
`--only grand` runs one search and skips the combination. And no memory cap is applied
during a dry run, because nothing is allocated.

**Regenerate it when a configuration changes, and not otherwise.** The store carries a
manifest naming the configs and the time, so a stale one is detectable rather than
merely suspected.

Three searches, all at the same `downsample_factor` so their masks are pixel-aligned:

| | config | what it asks |
|---|---|---|
| **GRAND alone** | `config/grand_arequipa_full.json` | 3–25° deployable ground seeing a target 10–40 km away, within ±3° of the horizon |
| **TAMBO alone** | `config/tambo_arequipa_full.json` | a 20–60° near wall facing a ≥25° far wall, 2–5 km across |
| **Combined** | `combine_experiments` over both | joint, union, and how much of each sits inside the other |

**What it costs.** 10204 × 12603 pixels, about 129 Mpx. At `downsample_factor: 4` the
estimator says 2.3 GiB against the ~6 GiB typically free; at 1 it says 4.5 GiB, which is
why 4 is the setting. That choice has a price worth stating: area is measured on the
downsampled mask while capacity is measured at full resolution, so a feature a few
pixels wide keeps its detectors and loses area. **Read these areas as lower bounds**,
and more so for TAMBO's canyon strips than for GRAND's blobs.

In [2]:
STORE = os.path.abspath(os.path.join("..", "results", "arequipa_full"))

def load_stored(label):
    """Reads one stored run, or returns None when the store does not have it yet."""
    path = os.path.join(STORE, f"{label}_results.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

manifest_path = os.path.join(STORE, "manifest.json")
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"store generated {manifest['generated']} by {manifest['generated_by']}")
    print(f"from {manifest['dem']}\n")
    for name in manifest["files"]:
        print("   ", name)
else:
    manifest = None
    print("The full-DEM store is empty: these searches have not been run yet.\n")
    print("Produce it with:")
    print("    python tools/run_arequipa_full.py --dry-run")
    print("    python tools/run_arequipa_full.py")

grand_full = load_stored("grand")
tambo_full = load_stored("tambo")

The full-DEM store is empty: these searches have not been run yet.

Produce it with:
    python tools/run_arequipa_full.py --dry-run
    python tools/run_arequipa_full.py


### GRAND over the whole DEM

The four things worth reading, in this order:

1. **The funnel**, and specifically whether the binding constraint is the same one the
   crops found. If a full DEM is bound by a different stage than its crops were, the
   crops were not representative and every number derived from them needs re-reading.
2. **The area**, against the crop scaled up — and against the closing factor this run
   reports for itself, rather than the 2.29× quoted from Colca.
3. **The site count and their spread.** A crop cannot say whether the good ground is one
   region or fifty scattered ones, and that is a deployment question, not a physics one.
4. **The weakest score component**, which on the crops is `solid_angle` everywhere. If
   that holds at full scale it is a statement about the criterion, not about Peru.

In [3]:
def summarise(results, label):
    if results is None:
        print(f"{label}: not in the store yet.")
        return
    chosen, shortlisted = explain.selected_sites(results)
    area = sum(s["area_km2"] for s in chosen)
    binding = explain.binding_constraint(results["funnel"])
    ratio = explain.closing_inflation(results["funnel"],
                                      results["parameters"]["candidate_stride"])
    weakest = [explain.weakest_component(s.get("arrival_scan") or {}) for s in chosen]
    named = [w[0] for w in weakest if w]

    print(f"{label}")
    print(f"   sites          {results['results']['total_sites']:>10,}"
          f"   ({len(shortlisted)} more cleared the thresholds, not selected)")
    print(f"   capacity       {results['results']['total_capacity']:>10,}")
    print(f"   area           {area:>10,.1f} km²")
    if binding:
        print(f"   bound by       {binding['stage']}"
              f"  (kept {100*binding['kept_fraction']:.1f}%)")
    if ratio is not None:
        print(f"   closing moved  {ratio:>10.2f}x")
    if named:
        commonest = max(set(named), key=named.count)
        print(f"   weakest        {commonest} at {named.count(commonest)}/{len(named)} sites")

summarise(grand_full, "GRAND, full Arequipa DEM")

GRAND, full Arequipa DEM: not in the store yet.


In [4]:
if grand_full is not None:
    print(grand_full.get("explanation") or explain.explain_results(grand_full))
else:
    print("Nothing stored for GRAND yet -- see the cell above for how to produce it.")

Nothing stored for GRAND yet -- see the cell above for how to produce it.


### TAMBO over the whole DEM

The more interesting of the two, because TAMBO's criteria are canyon-shaped and the
crop was *chosen* for containing a canyon. Over the whole DEM the question becomes: how
much other canyon is there, and is any of it as good?

Note that the areas here are the ones most affected by `downsample_factor: 4`: a strip
along a wall is exactly the feature that loses area to downsampling while keeping its
detectors.

In [5]:
summarise(tambo_full, "TAMBO, full Arequipa DEM")

TAMBO, full Arequipa DEM: not in the store yet.


In [6]:
if tambo_full is not None:
    print(tambo_full.get("explanation") or explain.explain_results(tambo_full))
else:
    print("Nothing stored for TAMBO yet -- see above for how to produce it.")

Nothing stored for TAMBO yet -- see above for how to produce it.


### Where both are viable

The overlay. On the Colca crop the answer was decided by slope: GRAND's 3–25° deployable
band against Colca's ~40° walls leaves only a 20–25° sliver, so the joint was about a
percent of GRAND's area and three fifths of TAMBO's.

Whether that survives at full scale is a real question. The crop contains one canyon
system; the DEM contains many, of varying wall slope, and the joint area is the
programme-level number — one site, one road, one power feed, two experiments.

In [7]:
report_path = os.path.join(STORE, "combined_report.json")
if not os.path.exists(report_path):
    print("The combination has not been produced yet.")
else:
    with open(report_path) as f:
        report = json.load(f)

    width = max(len(r["label"]) for r in report["runs"])
    print(f"   {'experiment'.ljust(width)} {'area km²':>12} {'sites':>7} "
          f"{'capacity':>10} {'in joint':>9}")
    print("   " + "-" * (width + 42))
    for r in report["runs"]:
        print(f"   {r['label'].ljust(width)} {r['area_km2']:>12,.1f} "
              f"{r['reported_sites']:>7,} {r['reported_capacity']:>10,} "
              f"{100*r['fraction_of_own_area_in_joint']:>8.1f}%")
    print("   " + "-" * (width + 42))
    print(f"   joint  {report['joint']['area_km2']:>10,.1f} km²")
    print(f"   union  {report['union']['area_km2']:>10,.1f} km²")
    for pair, stats in report["pairwise_overlap"].items():
        print(f"   {pair}: Jaccard {stats['jaccard']:.4f}")

The combination has not been produced yet.


And the overlay explains itself too, the same way a search does — including the part
that is easy to get wrong. Co-location is decided by whichever *ground* property the
two experiments share least of, because a pixel has one slope and both have to accept
it. What each asks of the **view** — the distance window, the arrival elevations — may
differ freely: two experiments can look out from the same hillside at different ranges
without conflict.

`oroscope-combine` prints this and saves it as `combination_explanation.txt`.

In [8]:
if not os.path.exists(report_path):
    print("The combination has not been produced yet.")
else:
    runs = {label: res for label, res in (("GRAND", grand_full), ("TAMBO", tambo_full))
            if res is not None}
    print(explain.explain_combination(report, runs))

The combination has not been produced yet.


### Comparing against the crop

The crop's numbers, for reference — GRAND 4580.2 km² in 1 site with 5317 detectors,
TAMBO 83.6 km² in 15 sites with 9717, joint 50.1 km². If the full DEM's binding
constraint or weakest component differs from these, the crop was not representative and
the comparison is the finding.

A crop is chosen because it is interesting. **A search over ground chosen for being
interesting is not a survey**, and that is the gap this run exists to close.

In [9]:
if grand_full is not None and tambo_full is not None:
    print("Both full-DEM runs are stored; compare their funnels against the crops':\n")
    for label, res in (("GRAND", grand_full), ("TAMBO", tambo_full)):
        b = explain.binding_constraint(res["funnel"])
        print(f"   {label:>6}: bound by {b['stage']!r} "
              f"(kept {100*b['kept_fraction']:.1f}%)")
    print("\n   Colca crop, for comparison:")
    print("   GRAND: bound by 'directions accepted' (kept 60.1%)")
    print("   TAMBO: bound by 'directions accepted' (kept 17.5%)")
else:
    print("Run both searches to make this comparison.")

Run both searches to make this comparison.


---

*Part of the [Oroscope](https://github.com/mbustama/oroscope) tutorials. Previous: [Combining and sensitivity](06_combining_and_sensitivity.ipynb). Full API reference: [oroscope docs](https://mbustama.github.io/oroscope/functions.html).*